In [ ]:
# =============================================================================
#  2048 — N-tuple TD-afterstate learning (Szubert/Jaskowski-style SOTA)
#
#  Replaces the previous DQN+ResNet pipeline. For 2048 specifically, N-tuple
#  networks with TD-afterstate learning are the established SOTA: linear
#  function approximation over patterns of board cells, indexed by their
#  log2 values, updated via afterstate TD(0). The literature consistently
#  reaches 32k tiles with this family of methods; deep RL plateaus much
#  earlier.
#
#  GPU is NOT used. The hot path is millions of small lookups, which the CPU
#  + Numba JIT handles far faster than a GPU.
# =============================================================================
import numpy as np
import numba
from numba import njit
import os, csv, time

# -----------------------------------------------------------------------------
# 1. Game engine: precomputed row lookup tables, log2 board representation.
# -----------------------------------------------------------------------------
def _build_move_tables():
    """For every 16-bit packed row (4 cells * 4 bits log2), precompute the
    move-left result row and the score gained."""
    move_table  = np.zeros(65536, dtype=np.uint16)
    score_table = np.zeros(65536, dtype=np.uint32)
    for row_val in range(65536):
        cells = [(row_val >> (4 * i)) & 0xF for i in range(4)]
        nz = [c for c in cells if c != 0]
        merged = []; score = 0; skip = False
        for j in range(len(nz)):
            if skip: skip = False; continue
            if j + 1 < len(nz) and nz[j] == nz[j + 1]:
                v = min(nz[j] + 1, 15); merged.append(v); score += 1 << v; skip = True
            else:
                merged.append(nz[j])
        merged += [0] * (4 - len(merged))
        new_val = 0
        for i, v in enumerate(merged):
            new_val |= (v & 0xF) << (4 * i)
        move_table[row_val] = new_val
        score_table[row_val] = score
    return move_table, score_table

MOVE_TABLE, SCORE_TABLE = _build_move_tables()


@njit(cache=True)
def move_jit(board, direction, MT, ST):
    """Apply a 2048 move (0=up,1=down,2=left,3=right). Returns (new_board, score, changed)."""
    new_board = np.zeros((4, 4), dtype=np.uint8)
    score = np.uint32(0); changed = False
    if direction == 2:
        for j in range(4):
            rv = (np.uint16(board[j,0]) | (np.uint16(board[j,1]) << 4) |
                  (np.uint16(board[j,2]) << 8) | (np.uint16(board[j,3]) << 12))
            nv = MT[rv]; score += ST[rv]
            new_board[j,0]=nv&0xF; new_board[j,1]=(nv>>4)&0xF
            new_board[j,2]=(nv>>8)&0xF; new_board[j,3]=(nv>>12)&0xF
            if rv != nv: changed = True
    elif direction == 3:
        for j in range(4):
            rv = (np.uint16(board[j,3]) | (np.uint16(board[j,2]) << 4) |
                  (np.uint16(board[j,1]) << 8) | (np.uint16(board[j,0]) << 12))
            nv = MT[rv]; score += ST[rv]
            new_board[j,3]=nv&0xF; new_board[j,2]=(nv>>4)&0xF
            new_board[j,1]=(nv>>8)&0xF; new_board[j,0]=(nv>>12)&0xF
            if rv != nv: changed = True
    elif direction == 0:
        for j in range(4):
            rv = (np.uint16(board[0,j]) | (np.uint16(board[1,j]) << 4) |
                  (np.uint16(board[2,j]) << 8) | (np.uint16(board[3,j]) << 12))
            nv = MT[rv]; score += ST[rv]
            new_board[0,j]=nv&0xF; new_board[1,j]=(nv>>4)&0xF
            new_board[2,j]=(nv>>8)&0xF; new_board[3,j]=(nv>>12)&0xF
            if rv != nv: changed = True
    else:  # down
        for j in range(4):
            rv = (np.uint16(board[3,j]) | (np.uint16(board[2,j]) << 4) |
                  (np.uint16(board[1,j]) << 8) | (np.uint16(board[0,j]) << 12))
            nv = MT[rv]; score += ST[rv]
            new_board[3,j]=nv&0xF; new_board[2,j]=(nv>>4)&0xF
            new_board[1,j]=(nv>>8)&0xF; new_board[0,j]=(nv>>12)&0xF
            if rv != nv: changed = True
    return new_board, score, changed


@njit(cache=True)
def spawn_tile_jit(board):
    """Spawn a 2 (90%) or 4 (10%) on a uniformly random empty cell. Mutates `board`."""
    cnt = 0
    for r in range(4):
        for c in range(4):
            if board[r, c] == 0: cnt += 1
    if cnt == 0: return False
    pick = np.random.randint(0, cnt); seen = 0
    for r in range(4):
        for c in range(4):
            if board[r, c] == 0:
                if seen == pick:
                    board[r, c] = 1 if np.random.random() < 0.9 else 2
                    return True
                seen += 1
    return False


# -----------------------------------------------------------------------------
# 2. N-tuple feature set: 4 base 6-tuples, closed under the D4 symmetry group
#    (4 rotations x {identity, hflip}). Yields 32 features sharing 4 weight
#    tables via parameter tying. Each table has 16^6 = 16,777,216 entries.
# -----------------------------------------------------------------------------
TUPLE_LEN  = 6
TABLE_SIZE = 16 ** TUPLE_LEN  # 16,777,216 entries per table

# Cells are flat indices 0..15 with index = row*4 + col.
BASE_PATTERNS_LIST = [
    [0, 1, 2, 3, 4, 5],     # row 0 + (1,0),(1,1)         "L-shape"
    [4, 5, 6, 7, 8, 9],     # row 1 + (2,0),(2,1)         shifted L
    [0, 1, 4, 5, 8, 9],     # 3x2 block in the top-left corner
    [0, 1, 5, 6, 9, 10],    # diagonal zigzag
]

def _gen_d4_symmetries(pattern_flat):
    """Generate the 8 D4-symmetric variants of a pattern (cells in order)."""
    cells = [(idx // 4, idx % 4) for idx in pattern_flat]
    sym = []
    for k in range(4):
        for flip in (False, True):
            new_cells = []
            for r, c in cells:
                rr, cc = r, c
                for _ in range(k):
                    rr, cc = cc, 3 - rr   # one CCW rotation: (r,c) -> (c, 3-r)
                if flip: cc = 3 - cc      # horizontal flip: (r,c) -> (r, 3-c)
                new_cells.append((rr, cc))
            sym.append([r_*4 + c_ for r_, c_ in new_cells])
    return sym

def _build_features():
    feats, ftable = [], []
    for ti, base in enumerate(BASE_PATTERNS_LIST):
        for sym in _gen_d4_symmetries(base):
            feats.append(sym); ftable.append(ti)
    return np.array(feats, dtype=np.int32), np.array(ftable, dtype=np.int32)

ALL_FEATURES, FEATURE_TO_TABLE = _build_features()
N_FEATURES = ALL_FEATURES.shape[0]   # 32
N_TABLES   = len(BASE_PATTERNS_LIST) # 4

WEIGHTS_MB = N_TABLES * TABLE_SIZE * 4 / 2**20
print(f"N-tuple network: {N_FEATURES} features over {N_TABLES} tables, {WEIGHTS_MB:.0f} MB weights")


# -----------------------------------------------------------------------------
# 3. JIT'd value, update, action selection, episode rollout.
#    V(s) = sum over 32 features of W[ftable[f], idx(f, s)].
#    TD-afterstate update for V on the prior afterstate s':
#        target = r' + V(s'')   where (s'', r') is the next afterstate + reward
#        V(s') <- V(s') + alpha * (target - V(s')) / N_FEATURES
#    Spread per-weight by N_FEATURES so the effective learning rate is alpha.
# -----------------------------------------------------------------------------
@njit(cache=True, inline='always')
def board_value(board, weights, feats, ftable):
    total = np.float32(0.0)
    n = feats.shape[0]; L = feats.shape[1]
    for f in range(n):
        idx = np.uint32(0)
        for k in range(L):
            cell = feats[f, k]
            v = np.uint32(board[cell // 4, cell % 4])
            idx = (idx << np.uint32(4)) | v
        total += weights[ftable[f], idx]
    return total


@njit(cache=True)
def board_update(board, weights, feats, ftable, delta):
    n = feats.shape[0]; L = feats.shape[1]
    for f in range(n):
        idx = np.uint32(0)
        for k in range(L):
            cell = feats[f, k]
            v = np.uint32(board[cell // 4, cell % 4])
            idx = (idx << np.uint32(4)) | v
        weights[ftable[f], idx] += delta


@njit(cache=True)
def select_best(board, MT, ST, weights, feats, ftable):
    """Greedy 1-ply lookahead: argmax over actions of (reward + V(afterstate))."""
    best_a = -1
    best_val = -np.float32(1e30)
    best_after = np.zeros((4, 4), dtype=np.uint8)
    best_reward = np.uint32(0)
    for d in range(4):
        nb, sc, ch = move_jit(board, d, MT, ST)
        if ch:
            v = np.float32(sc) + board_value(nb, weights, feats, ftable)
            if v > best_val:
                best_val = v; best_a = d; best_after = nb; best_reward = sc
    return best_a, best_after, best_reward


@njit(cache=True)
def play_episode(weights, feats, ftable, MT, ST, alpha):
    """Greedy self-play with on-line TD-afterstate learning."""
    board = np.zeros((4, 4), dtype=np.uint8)
    spawn_tile_jit(board); spawn_tile_jit(board)
    last_after = np.zeros((4, 4), dtype=np.uint8)
    has_last = False
    total_score = np.int64(0); n_moves = np.int64(0)
    n_feat = np.float32(feats.shape[0])
    while True:
        a, after, reward = select_best(board, MT, ST, weights, feats, ftable)
        if a == -1:
            # Terminal: V(last_after) target is 0 (no further reward).
            if has_last:
                lv = board_value(last_after, weights, feats, ftable)
                board_update(last_after, weights, feats, ftable, -lv * alpha / n_feat)
            break
        if has_last:
            cv = board_value(last_after, weights, feats, ftable)
            nv = board_value(after,      weights, feats, ftable)
            target = np.float32(reward) + nv
            board_update(last_after, weights, feats, ftable, (target - cv) * alpha / n_feat)
        last_after = after.copy()
        has_last = True
        for r in range(4):
            for c in range(4):
                board[r, c] = after[r, c]
        spawn_tile_jit(board)
        total_score += np.int64(reward); n_moves += 1
    mx = 0
    for r in range(4):
        for c in range(4):
            if board[r, c] > mx: mx = board[r, c]
    return mx, total_score, n_moves


@njit(cache=True)
def evaluate_episode(weights, feats, ftable, MT, ST):
    """Greedy 1-ply play with no learning — for fair test-time stats."""
    board = np.zeros((4, 4), dtype=np.uint8)
    spawn_tile_jit(board); spawn_tile_jit(board)
    score = np.int64(0)
    while True:
        a, after, reward = select_best(board, MT, ST, weights, feats, ftable)
        if a == -1: break
        for r in range(4):
            for c in range(4):
                board[r, c] = after[r, c]
        spawn_tile_jit(board)
        score += np.int64(reward)
    mx = 0
    for r in range(4):
        for c in range(4):
            if board[r, c] > mx: mx = board[r, c]
    return mx, score


# -----------------------------------------------------------------------------
# 4. Training loop with checkpointing + CSV logging.
# -----------------------------------------------------------------------------
ALPHA            = np.float32(0.1)
N_EPISODES       = 1_000_000
LOG_EVERY        = 200
SAVE_EVERY       = 10_000
EVAL_EVERY       = 50_000
EVAL_GAMES       = 200
checkpoint_path  = "ntuple_weights.npy"
log_path         = "training_log.csv"

# Resume if a checkpoint exists.
if os.path.exists(checkpoint_path):
    print(f"resuming from {checkpoint_path}")
    weights = np.load(checkpoint_path)
    if weights.shape != (N_TABLES, TABLE_SIZE) or weights.dtype != np.float32:
        print("  shape/dtype mismatch -- starting fresh")
        weights = np.zeros((N_TABLES, TABLE_SIZE), dtype=np.float32)
else:
    print(f"allocating fresh weights ({WEIGHTS_MB:.0f} MB)...")
    weights = np.zeros((N_TABLES, TABLE_SIZE), dtype=np.float32)

# CSV log: append, write header if new.
if not os.path.exists(log_path):
    with open(log_path, 'w', newline='') as f:
        csv.writer(f).writerow(['episode', 'max_tile', 'score',
                                'avg_max_log_500', 'rate_2048_500', 'rate_4096_500',
                                'eps_per_sec'])

print("compiling JIT (first call)...")
t0 = time.time()
play_episode(weights, ALL_FEATURES, FEATURE_TO_TABLE, MOVE_TABLE, SCORE_TABLE, ALPHA)
print(f"  done in {time.time()-t0:.1f}s")

print("training. Ctrl+C to stop; weights are saved every "
      f"{SAVE_EVERY} episodes.")

recent_max  = []
recent_score = []
t_start = time.time()
ep_offset = 0

try:
    for ep in range(N_EPISODES):
        mx, sc, mv = play_episode(
            weights, ALL_FEATURES, FEATURE_TO_TABLE,
            MOVE_TABLE, SCORE_TABLE, ALPHA
        )
        recent_max.append(mx)
        recent_score.append(int(sc))
        if len(recent_max) > 500:
            recent_max.pop(0); recent_score.pop(0)

        if (ep + 1) % LOG_EVERY == 0:
            window = recent_max
            avg_log = sum(window) / len(window)
            r2048 = sum(1 for m in window if m >= 11) / len(window)
            r4096 = sum(1 for m in window if m >= 12) / len(window)
            eps_per_s = (ep + 1) / (time.time() - t_start)
            tile = (1 << mx) if mx > 0 else 0
            print(f"\rep {ep+1:>7d} | last max {tile:>5d} | avg log {avg_log:5.2f} "
                  f"| 2048 {r2048:5.1%} | 4096 {r4096:5.1%} | {eps_per_s:5.0f} ep/s",
                  end="", flush=True)
            with open(log_path, 'a', newline='') as f:
                csv.writer(f).writerow([ep+1, tile, int(sc),
                                        f"{avg_log:.3f}",
                                        f"{r2048:.4f}", f"{r4096:.4f}",
                                        f"{eps_per_s:.1f}"])

        if (ep + 1) % SAVE_EVERY == 0:
            np.save(checkpoint_path, weights)

        if (ep + 1) % EVAL_EVERY == 0:
            print()  # newline before eval block
            print(f"  >>> eval @ episode {ep+1}: {EVAL_GAMES} games (no learning)")
            ev_max = []; ev_score = []
            te0 = time.time()
            for _ in range(EVAL_GAMES):
                em, es = evaluate_episode(weights, ALL_FEATURES, FEATURE_TO_TABLE,
                                          MOVE_TABLE, SCORE_TABLE)
                ev_max.append(em); ev_score.append(int(es))
            for tile_log in (10, 11, 12, 13, 14):
                r = sum(1 for m in ev_max if m >= tile_log) / EVAL_GAMES
                print(f"      >= {1 << tile_log:>5d}:  {r:6.1%}")
            print(f"      avg score: {sum(ev_score)/EVAL_GAMES:,.0f}   "
                  f"({EVAL_GAMES} games in {time.time()-te0:.1f}s)")
except KeyboardInterrupt:
    print("\ninterrupted; saving final checkpoint")

np.save(checkpoint_path, weights)
print(f"\nsaved {checkpoint_path}")


N-tuple network: 32 features over 4 tables, 256 MB weights
allocating fresh weights (256 MB)...
compiling JIT (first call)...
  done in 5.2s
training. Ctrl+C to stop; weights are saved every 10000 episodes.
ep   50000 | last max  4096 | avg log 11.38 | 2048 87.2% | 4096 51.4% |    73 ep/s
  >>> eval @ episode 50000: 200 games (no learning)
      >=  1024:   99.0%
      >=  2048:   85.0%
      >=  4096:   44.5%
      >=  8192:    0.5%
      >= 16384:    0.0%
      avg score: 47,899   (200 games in 2.9s)
ep  100000 | last max  2048 | avg log 11.68 | 2048 89.6% | 4096 69.4% |    57 ep/s
  >>> eval @ episode 100000: 200 games (no learning)
      >=  1024:   97.0%
      >=  2048:   90.5%
      >=  4096:   69.0%
      >=  8192:   10.5%
      >= 16384:    0.0%
      avg score: 66,758   (200 games in 3.2s)
ep  150000 | last max  2048 | avg log 12.01 | 2048 90.8% | 4096 77.6% |    48 ep/s
  >>> eval @ episode 150000: 200 games (no learning)
      >=  1024:   99.0%
      >=  2048:   93.5%
      